In [ ]:
import asyncio
from tqdm.asyncio import tqdm
import polars as pl
from langchain_community.vectorstores.neo4j_vector import Neo4jVector
from langchain_core.documents import Document
import sys
from tqdm.auto import tqdm
from neo4j import AsyncGraphDatabase, GraphDatabase
from dotenv import dotenv_values


def reformat_snomed(data_path: str) -> pl.LazyFrame:
        def _scan_active_snomed(filename: str) -> pl.LazyFrame:
            lf = pl.scan_csv(source=filename,
                            encoding="utf8",
                            separator="\t",
                            infer_schema_length=10_000,
                            quote_char=None
                            )
            lf = lf.filter(
                pl.col("active").cast(pl.Int8) == 1
            )
            return lf
        accept_tag = ["finding", "disorder", "situation", "morphologic abnormality", "procedure", "regime/therapy", 
                    "substance", "product", "medicinal product", "clinical drug", "organism", "body structure", 
                    "cell", "event", "observable entity", "record artifact", "physical object", 
                    "staging scale", "assessment scale", "physical force"]
        terms_path = data_path + "Snapshot/Terminology/sct2_Concept_Snapshot_INT_20250301.txt"
        descs_path = data_path + "Snapshot/Terminology/sct2_Description_Snapshot-en_INT_20250301.txt"
        relat_path = data_path + "Snapshot/Terminology/sct2_Relationship_Snapshot_INT_20250301.txt"
        active_terms = _scan_active_snomed(terms_path)
        active_descs = _scan_active_snomed(descs_path)
        active_relat = _scan_active_snomed(relat_path)

        active_concp = active_terms.join(active_descs, 
                                        left_on="id", 
                                        right_on="conceptId", 
                                        how="inner")
        active_concp = active_concp.rename({
                                        "id":"concept_id",
                                        "id_right": "term_id", 
                                        "term": "concept_name", 
                                        "typeId": "name_type"
                                        })

        active_synon = (
                    active_concp
                        .with_columns(
                            pl.col('name_type')
                                .replace_strict({
                                        900000000000003001: "FSN",
                                        900000000000013009: "Synonym"
                                        },
                                        default=None)
                            )
                        .drop_nulls('name_type')
                        .select([
                            "concept_id","name_type","concept_name"
                        ])
        )
        active_concp = (
                    active_synon
                        .filter(pl.col('name_type') == "FSN")
                        .with_columns(
                            pl.col('concept_name')
                            .str.extract(r"\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$")
                            .alias('semantic_tag'),
                            pl.lit("concept").alias("LABEL")
                        )
                        .drop('name_type')
                        .rename({"concept_name":"FSN"})
        )
        concepts = (
            pl.Series(active_concp
            .filter(
                pl.col("semantic_tag").is_in(accept_tag)
            )
            .select("concept_id").unique().collect(engine='streaming')).to_list()
        )
        active_synon = active_synon.filter(pl.col("concept_id").is_in(concepts))
        values = pl.Series(active_relat.select('typeId').unique().collect(engine='streaming')).to_list()
        mapping = (
            active_concp
            .filter(pl.col('concept_id').is_in(values))  # Filter by concept_id values
            .with_columns(
                pl.col('concept_id').cast(pl.Int64),  # Cast concept_id to Int64
                pl.col('FSN').cast(pl.Utf8)
                .str.replace(r"\((\w+\s?.?\s?\w+.?\w+.?\w+.?)\)$","")
                .str.strip_chars()
                
            )
            .select(["concept_id", "FSN"])  # Select the columns you need
        )
        mapping = (
            mapping
            .with_columns(
                  pl.col("FSN")
                  .str.replace(" ","_")
                  .str.to_lowercase()
            )   
            .collect(engine='streaming')  # Collect into a DataFrame
            .to_dict()  # Convert to dictionary
        )

        mapping_dict = dict(zip(mapping["concept_id"], mapping["FSN"]))
        active_relat = (
                    active_relat
                        .with_columns(
                            pl.col('typeId')
                            .replace_strict(
                                mapping_dict,
                                default = None
                            )
                            .alias('TYPE')
                        )
                        .drop_nulls('TYPE')               
        )
        relat = active_relat.select(['sourceId','destinationId','TYPE'])
        return active_concp, active_synon, relat



async def awrite_concept_nodes(
    lf: pl.LazyFrame,
    neo4j_url: str,
    neo4j_user: str,
    neo4j_password: str,
    batch_size: int = 10_000,
) -> None:
    """
    Asynchronously writes Concept nodes (with metadata) into Neo4j.
    Each row in `lf` should have: concept_id, FSN, semantic_tag, LABEL.
    """
    df = await asyncio.to_thread(lf.collect)
    total = df.height

    driver = AsyncGraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_password))
    async with driver:
        async with driver.session() as session:
            pbar = tqdm(total=total, desc="Writing Concept Nodes")
            start = 0

            while start < total:
                batch_df = await asyncio.to_thread(lambda: df[start : start + batch_size])

                rows = batch_df.select([
                    pl.col("concept_id"),
                    pl.col("FSN"),
                    pl.col("semantic_tag"),
                    pl.col("LABEL")
                ]).to_dicts()

                if rows:
                    # 6. Write batch in one UNWIND
                    cypher = """
                    UNWIND $rows AS r
                    MERGE (c:Concept {concept_id: r.concept_id})
                    SET c.FSN = r.FSN,
                        c.semantic_tag = r.semantic_tag,
                        c.LABEL = r.LABEL
                    """
                    await session.run(cypher, rows=rows)

                pbar.update(len(rows))
                start += batch_size

            pbar.close()


async def awrite_description_nodes(
    lf: pl.LazyFrame,
    embedding_model,
    neo4j_url: str,
    neo4j_user: str,
    neo4j_password: str,
    batch_size: int = 10_000,
) -> None:
    """
    Writes Description nodes (with embeddings) and
    links them to Concept nodes in Neo4j, in batches.
    """
    # 1. Clean & collect once
    lf_filtered = (
        lf
        .with_columns([
            pl.col("concept_name").cast(pl.Utf8).str.strip_chars(),
            pl.col("name_type")
              .cast(pl.Utf8)
              .str.strip_chars()
              .fill_null("Synonym")
        ])
        .drop_nulls(["concept_id", "concept_name"])
    )
    df_all = await asyncio.to_thread(lf_filtered.collect)
    total = df_all.height

    # 2. Init vector store once
    vector_store = await Neo4jVector.afrom_documents(
        documents=[],
        embedding=embedding_model,
        url=neo4j_url,
        username=neo4j_user,
        password=neo4j_password,
        database="neo4j",
        node_label="Description",
        embedding_node_property="embedding",
        text_node_property="text",
        index_name="name_embedding",
        pre_delete_collection=True,
    )

    # 3. Open Neo4j driver & session once
    driver = AsyncGraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_password))
    session = driver.session()

    # 4. Batch loop
    pbar = tqdm(total=total, desc="Writing Description Nodes")
    for start in range(0, total, batch_size):
        # a) slice and collect in thread
        batch_df = await asyncio.to_thread(lambda: df_all[start:start + batch_size])
        rows = batch_df.to_dicts()

        # b) build Document list
        documents = [
            Document(
                page_content=r["concept_name"],
                metadata={
                    "concept_id": r["concept_id"],
                    "name": r["concept_name"],
                    "type": r["name_type"],
                }
            )
            for r in rows
        ]

        # c) upload embeddings
        try:
            if documents:
                await vector_store.aadd_documents(documents)
        except Exception as e:
            print(f"[Batch {start}] Embedding upload error: {e}")

        # d) link descriptions to concepts
        #    use the same `text_node_property` ("text") to match Description nodes
        try:
            params = [
                {"concept_id": r["concept_id"], "text": r["concept_name"]}
                for r in rows
            ]
            await session.run(
                """
                UNWIND $batch AS row
                MATCH (c:Concept {concept_id: row.concept_id})
                MATCH (d:Description {text: row.text})
                MERGE (c)-[:HAS_NAME]->(d)
                """,
                batch=params
            )
        except Exception as e:
            print(f"[Batch {start}] Relationship merge error: {e}")

        # e) update progress
        pbar.update(len(rows))

    # 5. Cleanup
    pbar.close()
    await session.close()
    await driver.close()


async def awrite_relationships(
    lf: pl.LazyFrame,
    neo4j_url: str,
    neo4j_user: str,
    neo4j_password: str,
    batch_size: int = 10_000,
):
    """Writes dynamic relationships between Concept nodes to Neo4j using apoc.create.relationship."""
    try:
        total_rows = await asyncio.to_thread(lambda: lf.select(pl.len()).collect().item())
        pbar = tqdm(total=total_rows, desc="Writing Relationships")
        start = 0

        # Initialize Neo4j driver
        driver = AsyncGraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_password))
        async with driver:
            async with driver.session() as session:
                while True:
                    batch_lf = lf.slice(offset=start, length=batch_size)
                    rows = await asyncio.to_thread(lambda: batch_lf.select(pl.len()).collect().item())
                    batch_data = batch_lf.collect().to_dicts()

                    relationships = [
                        {
                            "sourceId": row.get("sourceId"),
                            "destinationId": row.get("destinationId"),
                            "TYPE": row.get("TYPE"),
                        }
                        for row in batch_data
                        if row.get("sourceId") and row.get("destinationId") and row.get("TYPE")
                    ]

                    if relationships:
                        query = """
                        UNWIND $batch AS row
                        MATCH (source:Concept {concept_id: row.sourceId})
                        MATCH (target:Concept {concept_id: row.destinationId})
                        CALL apoc.create.relationship(source, row.TYPE, {}, target)
                        YIELD rel
                        RETURN count(*)
                        """
                        await session.run(query, parameters={"batch": relationships})

                    start += batch_size
                    pbar.update(rows)
                    if rows < batch_size:
                        break
        pbar.close()

    except Exception as e:
        print(f"Error writing relationships: {e}")




c:\Users\Natt\miniconda3\envs\qd_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
env = dotenv_values(r"H:\NatT\snomed\.env")
neo4j_dict = {
        "uri": env.get('NEO4J_HOST'),
        "username": env.get('NEO4J_USER'),
        "password": env.get('NEO4J_PASSWORD'),
        "snomed_release": env.get('SNOMED_RELEASE_PATH'),
}
neo4j_uri = neo4j_dict["uri"]
neo4j_username = neo4j_dict["username"]
neo4j_secret = neo4j_dict["password"]

In [ ]:
from models import (
    AsyncPubMedBERTEmbedder, 
    AsyncSapBERTEmbedder, 
    AsyncSapUMLSEmbedder,
    AsyncBioClinicalBERTEmbedder, 
    AsyncGatorTronEmbedder,
    AsyncSapBERTpoolEmbedder)

In [ ]:
path = neo4j_dict["snomed_release"]
c_, s_, r_ = reformat_snomed(data_path=path)

In [ ]:
async def main():
      if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
      await awrite_concept_nodes(
          lf=c_, 
          neo4j_url=neo4j_uri, 
          neo4j_user=neo4j_username, 
          neo4j_password=neo4j_secret,
          batch_size = 10_000,
          )
      await awrite_description_nodes(
          lf= s_,
          embedding_model=embedding_model,
          neo4j_url=neo4j_uri,
          neo4j_user=neo4j_username,
          neo4j_password=neo4j_secret,
          batch_size = 10_000,
      )
      await awrite_relationships(r_, neo4j_uri, neo4j_username, neo4j_secret)


In [ ]:
def delete_all_in_batches(uri, user, password, database="neo4j", batch_size=30000, pause_sec=0.0):
    """
    Repeatedly runs:
        MATCH (n) WITH n LIMIT $batch DETACH DELETE n
    until no more nodes are deleted.

    Returns a dict with total deleted counts and loop iterations.
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))
    total_nodes = 0
    total_rels = 0
    loops = 0

    try:
        with driver.session(database=database) as session:
            while True:
                result = session.run(
                    "MATCH (n) WITH n LIMIT $batch DETACH DELETE n",
                    batch=batch_size,
                )
                summary = result.consume()
                deleted_nodes = summary.counters.nodes_deleted
                deleted_rels = summary.counters.relationships_deleted
                loops += 1
                total_nodes += deleted_nodes
                total_rels += deleted_rels

                if deleted_nodes == 0:
                    break
                if pause_sec > 0:
                    import time
                    time.sleep(pause_sec)
    finally:
        driver.close()

    return {"nodes_deleted": total_nodes, "relationships_deleted": total_rels, "loops": loops}

In [6]:
def topk_accuracy_from_vector_search(
    gold_data,
    pred_data,
    vector_index,
    top_k_values=[1, 5, 10],
):
    """
    Evaluate top-k accuracy using vector similarity search.
    It uses the formal_term in predicted entities to query Neo4j vector index.

    Args:
        gold_data: list of dicts with 'entity' and 'Target code'
        pred_data: list of predicted entities (no 'tops', uses formal_term)
        vector_index: a Neo4jVector index instance (e.g., from_existing_index)
        top_k_values: list of k values to evaluate (default: [1, 5, 10])

    Returns:
        dict with top-k accuracy scores
    """
    assert len(gold_data) == len(pred_data), "Mismatch in number of records."

    topk_hits = {k: 0 for k in top_k_values}
    total_evaluated_spans = 0

    for gold_item, pred_entities in zip(gold_data, pred_data):
        gold_ents = {
            (e["start_index"], e["end_index"]): str(e["Target code"])
            for e in gold_item.get("entity", [])
        }

        
        # Keep track of spans processed in this record
        record_evaluated_spans = 0

        for pred in pred_entities:
            span = (pred["start_index"], pred["end_index"])
            if span not in gold_ents:
                continue

            gold_cid = gold_ents[span]
            query_text = pred.get("formal_term") or pred.get("term")

            if not query_text:
                continue

            # Vector search from Neo4j
            results = vector_index.similarity_search(query_text, k=max(top_k_values))
            top_candidates = [str(doc.metadata.get("concept_id")) for doc in results]
            
            # This is where we count the successes and failures for this span
            total_evaluated_spans += 1
            record_evaluated_spans += 1

            for k in top_k_values:
                if gold_cid in top_candidates[:k]:
                    topk_hits[k] += 1

    # FINAL SUMMARY LOG
#    print("\n" + "=" * 30)
#    print("      FINAL ACCURACY REPORT      ")
#    print("=" * 30)
    print(f"Total evaluated spans: {total_evaluated_spans}")

    results = {
        f"top{k}": topk_hits[k] / total_evaluated_spans if total_evaluated_spans else 0.0
        for k in top_k_values
    }

    # Format and print final results
    final_results_formatted = {
        f"TOP{k}": f"{(results[f'top{k}'] * 100):.2f}%"
        for k in top_k_values
    }
    
    #for k, v in final_results_formatted.items():
    #    print(f"  - {k} Accuracy: {v}")

    return final_results_formatted


In [ ]:
import json
gold_path = "DATA/test20.json"
gemini_path = "DATA/predictions/gemini_pro20_20250721.json"
claude_path = "DATA/predictions/claude_20_20250721.json"
gpt_path = "DATA/predictions/gpt41_20_20250721.json"

with open(gold_path, "r", encoding="utf-8") as f:
    gold_data = json.load(f)
with open(gemini_path, "r", encoding="utf-8") as f:
    gemini_data = json.load(f)
with open(gpt_path, "r", encoding="utf-8") as f:
    gpt_data = json.load(f)
with open(claude_path, "r", encoding="utf-8") as f:
    claude_data = json.load(f)

models = [
    ("Gemini", gemini_data),
    ("Claude", claude_data),
    ("GPT", gpt_data),
]

In [ ]:
embedding_model = AsyncPubMedBERTEmbedder()
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)

In [ ]:
embedding_model = AsyncSapUMLSEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncBioClinicalBERTEmbedder()
await main()


In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncGatorTronEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncSapBERTpoolEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncSapBERTEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)